WEEK 1 Group Project: Alder105 Group

# Week 1 Group Project — p_factor Prediction

**Group:** `Alder105`

## Summary

We predicted the `p_factor` of PNC test participants from **age, sex, and maternal education** using ridge regression with quadratic terms.

**Cross-validated performance on the training set: r = 0.298**

Our final model contains **no neuroimaging data**. We tested cortical thickness and surface area across 200 Schaefer parcels with seven model families (ridge, lasso,
elastic net, random forest, gradient boosting, SVR, PLS) plus PCA dimensionality reduction. In every case, structural brain features added nothing beyond age and sex,
and adding them to the demographic model *reduced* accuracy (0.298 → 0.242).

See `exploration.ipynb` for the full analysis supporting this decision. This notebook contains only the clean, reproducible final model and runs from top to bottom without modification.

**Note on the holdout:** the RBC phenotype file (`PNC_participants.tsv`) contains p_factor values for all participants including the test set. We never loaded it. All labels used here come from the instructor-provided `train_participants.tsv`.

In [28]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import KFold, cross_val_predict

GROUP_NAME = "Alder105"          
RBC_DIR = Path("/Users/kathleenomalley/Neurohack105_week1project/data")
# ---------------------------------------------------------------------------

RANDOM_STATE = 0
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)

In [29]:
train_data = pd.read_csv(RBC_DIR / "train_participants.tsv", sep="\t")
test_data = pd.read_csv(RBC_DIR / "test_participants.tsv", sep="\t")

print(f"train: {train_data.shape}")
print(f"test:  {test_data.shape}")
print("p_factor in test?", "p_factor" in test_data.columns)

train: (1280, 15)
test:  (321, 14)
p_factor in test? False


In [30]:
EDU_MAP = {
    "No/incomplete primary": 0,
    "Complete primary": 1,
    "Complete secondary": 2,
    "Complete tertiary": 3,
}

FEATURES = ["age", "sex_num", "p1_edu"]


def add_features(df):
    """Add the model's predictor columns to a participant dataframe."""
    df = df.copy()
    df["sex_num"] = (df["sex"] == "Male").astype(int)
    df["p1_edu"] = df["parent_1_education"].map(EDU_MAP)
    return df


train_data = add_features(train_data)
test_data = add_features(test_data)

# Sanity check: every non-null education value mapped to a number.
unmapped = train_data["parent_1_education"].notna() & train_data["p1_edu"].isna()
assert not unmapped.any(), f"unmapped education levels: {train_data.loc[unmapped, 'parent_1_education'].unique()}"

print("training missingness:")
print(train_data[FEATURES].isna().sum())

training missingness:
age         0
sex_num     0
p1_edu     19
dtype: int64


In [31]:
def make_model():
    return make_pipeline(
        PolynomialFeatures(degree=2),
        StandardScaler(),
        Ridge(alpha=1.0),
    )

In [32]:
fit_rows = train_data.dropna(subset=FEATURES + ["p_factor"])
print(f"fitting on {len(fit_rows)} of {len(train_data)} training participants")

pred_cv = cross_val_predict(make_model(), fit_rows[FEATURES], fit_rows["p_factor"], cv=cv)
r_overall = np.corrcoef(pred_cv, fit_rows["p_factor"])[0, 1]

# Per-fold r, to show the spread.
fold_rs = []
for tr_i, te_i in cv.split(fit_rows):
    m = make_model()
    m.fit(fit_rows.iloc[tr_i][FEATURES], fit_rows.iloc[tr_i]["p_factor"])
    p = m.predict(fit_rows.iloc[te_i][FEATURES])
    fold_rs.append(np.corrcoef(p, fit_rows.iloc[te_i]["p_factor"])[0, 1])
fold_rs = np.array(fold_rs)

print(f"\nCV r (pooled):  {r_overall:.3f}")
print(f"per-fold r:     {fold_rs.round(3)}")
print(f"fold mean:      {fold_rs.mean():.3f}")
print(f"fold SD:        {fold_rs.std():.3f}")

fitting on 1261 of 1280 training participants

CV r (pooled):  0.298
per-fold r:     [0.242 0.33  0.307 0.354 0.284]
fold mean:      0.303
fold SD:        0.039


In [33]:
model = make_model()
model.fit(fit_rows[FEATURES], fit_rows["p_factor"])

train_medians = fit_rows[FEATURES].median()
test_features = test_data[FEATURES].fillna(train_medians)

test_data["p_factor"] = model.predict(test_features)

print(f"predicted {len(test_data)} test participants")
print(f"NaN predictions: {test_data['p_factor'].isna().sum()}")
print(f"\nprediction range: {test_data['p_factor'].min():.3f} to {test_data['p_factor'].max():.3f}")
print(f"training p_factor range: {fit_rows['p_factor'].min():.3f} to {fit_rows['p_factor'].max():.3f}")
test_data.head()

predicted 321 test participants
NaN predictions: 0

prediction range: -1.025 to 0.691
training p_factor range: -1.608 to 2.543


,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,sex_num,p1_edu,p_factor
0,2285120424,PNC,PNC1,PNC1,1,16.166667,Female,Black,not Hispanic or Latino,22.67,Left,9th Grade,Complete secondary,Complete secondary,0,2.0,-0.469106
1,1854364375,PNC,PNC1,PNC1,1,16.833333,Female,White,not Hispanic or Latino,28.19,Right,10th Grade,Complete secondary,Complete primary,0,2.0,-0.401587
2,1448081953,PNC,PNC1,PNC1,1,19.583333,Female,White,not Hispanic or Latino,25.10,Right,Some College,Complete tertiary,Complete tertiary,0,3.0,-0.293197
3,1342685465,PNC,PNC1,PNC1,1,17.166667,Female,Black,not Hispanic or Latino,31.93,Right,10th Grade,Complete primary,Complete primary,0,1.0,-0.207563
4,3289562482,PNC,PNC1,PNC1,1,21.166667,Female,White,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,0,3.0,-0.058649


In [34]:
assert GROUP_NAME != "GROUP_NAME", "Set GROUP_NAME in the config cell before saving!"

results_dir = Path("/Users/kathleenomalley/Neurohack105_week1project/results")
results_dir.mkdir(exist_ok=True)
outfile = results_dir / f"{GROUP_NAME}.tsv"

test_data.to_csv(outfile, sep="\t", index=False)
print(f"wrote {outfile}")

# Verify the file we just wrote.
check = pd.read_csv(outfile, sep="\t")
n_expected = len(pd.read_csv(RBC_DIR / "test_participants.tsv", sep="\t"))
assert len(check) == n_expected, f"expected {n_expected} rows, got {len(check)}"
assert "participant_id" in check.columns, "missing participant_id column"
assert "p_factor" in check.columns, "missing p_factor column"
assert check["p_factor"].notna().all(), "some p_factor predictions are NaN"
print(f"verified: {len(check)} rows, both required columns present, no NaNs")

wrote /Users/kathleenomalley/Neurohack105_week1project/results/Alder105.tsv
verified: 321 rows, both required columns present, no NaNs


In [39]:
cd /Users/kathleenomalley/Neurohack105_week1project
git add alder105_final_submission.ipynb results/Alder105.tsv
git commit -m "Add Alder105 p_factor predictions: age + sex + maternal education, CV r = 0.298"
git push fork HEAD

SyntaxError: invalid syntax (2496316503.py, line 2)

In [37]:
ls


alder105_final_submission.ipynb  test.ipynb


In [38]:
git status

SyntaxError: invalid syntax (3528599804.py, line 1)